In [1]:
year = 2007
month = 1

In [2]:
# Parameters
year = 1999
month = 12


In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap
import calendar
import datetime
from datetime import date

### URLs

In [4]:
# Ufiles = "https://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Ufiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Vfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridV"
Wfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridW"
Tfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridT"
Sfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridS"
# #mesh url
# Zgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_zgr.nc"
# Hgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_hgr.nc"

### Environment 

In [5]:
os.environ["NETRC"] = "/home/b/b383184/.netrc"

### Mesh

In [6]:
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Zgr

<xarray.Dataset> Size: 555MB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 26MB ...
    hdept         (t, y, x) float64 106MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 106MB ...
    e3w_ps        (t, y, x) float64 106MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t

In [7]:
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')
ds_Hgr

<xarray.Dataset> Size: 1GB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/21)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    glamt         (t, y, x) float32 53MB ...
    glamu         (t, y, x) float32 53MB ...
    ...            ...
    e1f           (t, y, x) float64 106MB ...
    e2t           (t, y, x) float64 106MB ...
    e2u           (t, y, x) float64 106MB ...
    e2v           (t, y, x) float64 106MB ...
    e2f           (t, y, x) float64 106MB ...
    ff            (t, y, x) float64 106MB ...
Attributes:
    file_name:            mesh_hgr.nc
    TimeStamp:            03/02/2016 10:24:55 -0000
    Unlimited_Dimension:  t

### Functions

In [8]:
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

x0, x1, y0, y1

(2305, 3565, 1374, 1873)

In [9]:
last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)

print(end_date.strftime("%Y-%m-%d"))

1999-12-31


In [10]:
def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))

In [11]:
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file):

    from tqdm import tqdm
    import xarray as xr
    
    parts = []
    for tt in tqdm(range(len(days)//2)):
        da = (
            xr.open_dataset(url, engine="pydap", mask_and_scale=False, decode_cf=True)[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all.to_dataset(name=varname).to_netcdf(output_file, unlimited_dims=["time_counter"])
    print(f"Saved {output_file}")

In [12]:
starts = days[0::2]
ends = days[1::2].tolist()  
ends[-1] = days[-1]
ends

for tt in  range(len(days)//2):
    print('start_date '+str(starts[tt]))
    print('end_date '+str(ends[tt]))

start_date 1999-12-01 12:00:00
end_date 1999-12-02 12:00:00
start_date 1999-12-03 12:00:00
end_date 1999-12-04 12:00:00
start_date 1999-12-05 12:00:00
end_date 1999-12-06 12:00:00
start_date 1999-12-07 12:00:00
end_date 1999-12-08 12:00:00
start_date 1999-12-09 12:00:00
end_date 1999-12-10 12:00:00
start_date 1999-12-11 12:00:00
end_date 1999-12-12 12:00:00
start_date 1999-12-13 12:00:00
end_date 1999-12-14 12:00:00
start_date 1999-12-15 12:00:00
end_date 1999-12-16 12:00:00
start_date 1999-12-17 12:00:00
end_date 1999-12-18 12:00:00
start_date 1999-12-19 12:00:00
end_date 1999-12-20 12:00:00
start_date 1999-12-21 12:00:00
end_date 1999-12-22 12:00:00
start_date 1999-12-23 12:00:00
end_date 1999-12-24 12:00:00
start_date 1999-12-25 12:00:00
end_date 1999-12-26 12:00:00
start_date 1999-12-27 12:00:00
end_date 1999-12-28 12:00:00
start_date 1999-12-29 12:00:00
end_date 1999-12-31 12:00:00


### Data download

In [13]:
U_out = f'U_{start_date.strftime("%Y-%m")}.nc'
V_out = f'V_{start_date.strftime("%Y-%m")}.nc'
W_out = f'W_{start_date.strftime("%Y-%m")}.nc'
T_out = f'T_{start_date.strftime("%Y-%m")}.nc'
S_out = f'S_{start_date.strftime("%Y-%m")}.nc'

outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables/'

In [14]:
#U 
download_MERCATOR(
    Ufiles, "vozocrtx", starts, ends, x0, x1, y0, y1,outpath+U_out
)

  0%|                                                          | 0/15 [00:00<?, ?it/s]

  7%|███▎                                             | 1/15 [02:28<34:41, 148.65s/it]

 13%|██████▌                                          | 2/15 [04:30<28:48, 132.98s/it]

 20%|█████████▊                                       | 3/15 [05:41<20:55, 104.60s/it]

 27%|█████████████▎                                    | 4/15 [06:05<13:19, 72.67s/it]

...

 87%|██████████████████████████████████████████▍      | 13/15 [09:20<00:44, 22.37s/it]

 93%|█████████████████████████████████████████████▋   | 14/15 [09:45<00:23, 23.21s/it]

100%|█████████████████████████████████████████████████| 15/15 [10:13<00:00, 24.56s/it]

100%|█████████████████████████████████████████████████| 15/15 [10:13<00:00, 40.87s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/U_1999-12.nc


In [15]:
download_MERCATOR(
    Vfiles, "vomecrty", starts, ends, x0, x1, y0, y1,outpath+V_out 
)

  0%|                                                          | 0/15 [00:00<?, ?it/s]

  7%|███▎                                             | 1/15 [01:59<27:58, 119.93s/it]

 13%|██████▋                                           | 2/15 [02:19<13:09, 60.71s/it]

 20%|██████████                                        | 3/15 [02:37<08:17, 41.47s/it]

 27%|█████████████▎                                    | 4/15 [02:56<05:59, 32.67s/it]

...

 87%|██████████████████████████████████████████▍      | 13/15 [06:05<00:41, 20.55s/it]

 93%|█████████████████████████████████████████████▋   | 14/15 [06:27<00:21, 21.15s/it]

100%|█████████████████████████████████████████████████| 15/15 [06:59<00:00, 24.21s/it]

100%|█████████████████████████████████████████████████| 15/15 [06:59<00:00, 27.94s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/V_1999-12.nc


In [16]:
download_MERCATOR(
    Wfiles, "vovecrtz", starts, ends, x0, x1, y0, y1,outpath+W_out 
)

  0%|                                                          | 0/15 [00:00<?, ?it/s]

  7%|███▎                                              | 1/15 [00:19<04:36, 19.76s/it]

 13%|██████▋                                           | 2/15 [00:38<04:07, 19.06s/it]

 20%|██████████                                        | 3/15 [01:03<04:23, 21.92s/it]

 27%|█████████████▎                                    | 4/15 [02:31<08:46, 47.83s/it]

...

 87%|██████████████████████████████████████████▍      | 13/15 [06:30<01:05, 32.53s/it]

 93%|█████████████████████████████████████████████▋   | 14/15 [06:55<00:30, 30.25s/it]

100%|█████████████████████████████████████████████████| 15/15 [07:24<00:00, 29.73s/it]

100%|█████████████████████████████████████████████████| 15/15 [07:24<00:00, 29.62s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/W_1999-12.nc


In [17]:
download_MERCATOR(
    Tfiles, "votemper", starts, ends, x0, x1, y0, y1,outpath+T_out
)

  0%|                                                          | 0/15 [00:00<?, ?it/s]

  7%|███▎                                             | 1/15 [02:44<38:20, 164.30s/it]

 13%|██████▋                                           | 2/15 [03:12<18:16, 84.37s/it]

 20%|██████████                                        | 3/15 [03:36<11:17, 56.49s/it]

 27%|█████████████▎                                    | 4/15 [03:58<07:54, 43.09s/it]

...

 87%|██████████████████████████████████████████▍      | 13/15 [06:57<00:43, 21.62s/it]

 93%|█████████████████████████████████████████████▋   | 14/15 [07:19<00:21, 21.53s/it]

100%|█████████████████████████████████████████████████| 15/15 [07:45<00:00, 22.98s/it]

100%|█████████████████████████████████████████████████| 15/15 [07:45<00:00, 31.04s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/T_1999-12.nc


In [18]:
download_MERCATOR(
    Sfiles, "vosaline", starts, ends, x0, x1, y0, y1,outpath+S_out 
)

  0%|                                                          | 0/15 [00:00<?, ?it/s]

  7%|███▎                                             | 1/15 [02:54<40:45, 174.69s/it]

 13%|██████▋                                           | 2/15 [03:15<18:11, 83.97s/it]

 20%|██████████                                        | 3/15 [03:31<10:35, 52.98s/it]

 27%|█████████████▎                                    | 4/15 [03:57<07:47, 42.47s/it]

...

 87%|██████████████████████████████████████████▍      | 13/15 [06:58<00:39, 19.78s/it]

 93%|█████████████████████████████████████████████▋   | 14/15 [07:16<00:19, 19.19s/it]

100%|█████████████████████████████████████████████████| 15/15 [07:39<00:00, 20.59s/it]

100%|█████████████████████████████████████████████████| 15/15 [07:39<00:00, 30.66s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/S_1999-12.nc
